# PyDI Data Integration Workflow: Videogames

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with vidoegame datasets to showcase the data integration pipeline from schema and entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Schema Matching and Value Normalization](#part-1-schema-matching-and-value-normalization)
- [Part 2: Data Profiling](#part-2-data-profiling)
- [Part 3: Entity Matching](#part-3-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 4: Data Fusion](#part-4-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

### Datasets

- **DBpedia**: 65,000 records
- **Metacritic**: 20,494 records
- **Global Sales Ranking**: 7,877 records

## Part 1: Schema Matching and Value Normalization

In [83]:
from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input" / "games"
OUTPUT_DIR = NOTEBOOK_DIR / "output" / "games"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [84]:
import pandas as pd
import json
from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaTranslator
from PyDI.normalization import load_normalization_spec
from langchain_openai import ChatOpenAI

## Step 1: Load Target Schema and Normalization Spec

In [85]:
# Load the JSON Schema (used for both matching and normalization)
with open(INPUT_DIR / "schemamatching" / "target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the same schema
spec = load_normalization_spec(INPUT_DIR / "schemamatching" / "target_schema.json")

# Set genres column manually to list type
spec.set_column("genres", output_type="list")

target_columns = list(spec.columns.keys())

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

# Show column types derived from schema
pd.DataFrame([
    {"column": col, "output_type": col_spec.output_type}
    for col, col_spec in spec.columns.items()
])

,column,output_type
0,id,string
1,name,string
2,releaseYear,datetime
3,developer,string
4,publisher,string
5,platform,string
6,criticScore,float
7,userScore,float
8,ESRB,string
9,globalSales,int


## Step 2: Load Source Datasets

In [86]:
from PyDI.io import load_xml, load_csv, load_json
dbpedia = load_xml(INPUT_DIR / "schemamatching" / "original_data" / "dbpedia.xml")
dbpedia.attrs["dataset_name"] = "dbpedia"
dbpedia.head()

,gameLabel,platform,developer,genre,releaseDate,series
0,San Francisco Rush 2049,Game Boy Color,Handheld Games,Racing video game,2006-02-17,Rush (video game series)
1,RoboCop (1988 video game),Arcade video game,Ocean Software,Beat 'em up,1989-12-12,List of RoboCop video games
2,Air (video game),PlayStation Vita,Key (company),Eroge,2016-09-08,NaN
3,Fallout 2,Mac OS X,Black Isle Studios,Role-playing video game,1998-10-29,Fallout (series)
4,SpongeBob SquarePants: Creature from the Krust...,Wii,Blitz Games,Platform game,2006-10-18,SpongeBob SquarePants video games


In [87]:
metacritic = load_csv(INPUT_DIR / "schemamatching" / "original_data" / "metacritic.csv")
metacritic.attrs["dataset_name"] = "metacritic"
metacritic.head()

,name,release_date,developer,platform,genres,number_of_players,rating,metascore,user_score
0,Red Dead Redemption 2,"Oct 26, 2018",Rockstar Games,Xbox One,"Action Adventure,Open-World",Up to 32,M,97.0,8.3
1,Grand Theft Auto IV,"Apr 29, 2008",Rockstar North,Xbox 360,"Action Adventure,Modern,Modern,Open-World",1 Player,M,98.0,8.0
2,SoulCalibur,"Sep 8, 1999",Namco,Dreamcast,"Action,Fighting,3D",1-2,T,98.0,8.4
3,Tony Hawk's Pro Skater 2,"Sep 20, 2000",Neversoft Entertainment,PlayStation,"Sports,Alternative,Skateboarding",1-2,T,98.0,7.5
4,Super Mario Galaxy,"Nov 12, 2007",Nintendo,Wii,"Action,Platformer,Platformer,3D,3D",No Online Multiplayer,E,97.0,9.1


In [88]:
sales = load_json(INPUT_DIR / "schemamatching" / "original_data" / "sales.json")
sales.attrs["dataset_name"] = "sales"
sales.head()

,Title,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76,51,8,322.0,Nintendo,E
1,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82,73,8.3,709.0,Nintendo,E
2,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80,73,8,192.0,Nintendo,E
3,New Super Mario Bros.,DS,2006,Platform,Nintendo,11.28,9.14,6.50,2.88,29.80,89,65,8.5,431.0,Nintendo,E
4,Wii Play,Wii,2006,Misc,Nintendo,13.96,9.18,2.93,2.84,28.92,58,41,6.6,129.0,Nintendo,E


## Step 3: LLM-Based Schema Matching

In [89]:
from dotenv import load_dotenv
load_dotenv()

# Initialize matcher with target schema for better context
matcher = LLMBasedSchemaMatcher(
    chat_model=ChatOpenAI(model="gpt-5"),
    num_rows=40,
    target_schema=target_schema,
)

# Match dbpedia dataset
dbpedia_mapping = matcher.match(dbpedia, df_target)

dbpedia_mapping

[INFO ] PyDI.schemamatching.llm_based - Initialized LLMBasedSchemaMatcher with 40 sample rows
[INFO ] PyDI.schemamatching.llm_based - LLM-based schema matching: dbpedia -> target_schema
[INFO ] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO ] PyDI.utils.llm - LLM call: {"timestamp": "2025-12-12T15:31:57.695037Z", "row_index": 0, "attempt": 0, "provider_class": "ChatOpenAI", "model": "gpt-5", "duration_ms": 34968.860149383545, "temperature": 0.0, "max_tokens": null, "usage": {"input_tokens": 2423, "output_tokens": 374, "total_tokens": 2797, "input_token_details": {"audio": 0, "cache_read": 0}, "output_token_details": {"audio": 0, "reasoning": 320}}, "request_messages": [{"type": "system", "content": "You are an expert at aligning table schemas.\n\nGiven Table A and Table B, identify the matching columns between them. For every column in Table A, specify the corresponding column in Table B. If a column in Table A has no match in Table B, map

,source_dataset,source_column,target_dataset,target_column,score,notes
0,dbpedia,gameLabel,target_schema,name,0.95,llm_based_matching
1,dbpedia,platform,target_schema,platform,0.95,llm_based_matching
2,dbpedia,developer,target_schema,developer,0.95,llm_based_matching
3,dbpedia,genre,target_schema,genres,0.95,llm_based_matching
4,dbpedia,releaseDate,target_schema,releaseYear,0.95,llm_based_matching
5,dbpedia,series,target_schema,series,0.95,llm_based_matching


In [90]:
metacritic_mapping = matcher.match(metacritic, df_target)
metacritic_mapping

[INFO ] PyDI.schemamatching.llm_based - LLM-based schema matching: metacritic -> target_schema
[INFO ] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO ] PyDI.utils.llm - LLM call: {"timestamp": "2025-12-12T15:32:10.802542Z", "row_index": 0, "attempt": 0, "provider_class": "ChatOpenAI", "model": "gpt-5", "duration_ms": 13055.038928985596, "temperature": 0.0, "max_tokens": null, "usage": {"input_tokens": 3113, "output_tokens": 590, "total_tokens": 3703, "input_token_details": {"audio": 0, "cache_read": 0}, "output_token_details": {"audio": 0, "reasoning": 512}}, "request_messages": [{"type": "system", "content": "You are an expert at aligning table schemas.\n\nGiven Table A and Table B, identify the matching columns between them. For every column in Table A, specify the corresponding column in Table B. If a column in Table A has no match in Table B, map it to null. Represent each mapping as a two-item list like [\"Table A column\", \"Table B 

,source_dataset,source_column,target_dataset,target_column,score,notes
0,metacritic,name,target_schema,name,0.95,llm_based_matching
1,metacritic,release_date,target_schema,releaseYear,0.95,llm_based_matching
2,metacritic,developer,target_schema,developer,0.95,llm_based_matching
3,metacritic,platform,target_schema,platform,0.95,llm_based_matching
4,metacritic,genres,target_schema,genres,0.95,llm_based_matching
5,metacritic,rating,target_schema,ESRB,0.95,llm_based_matching
6,metacritic,metascore,target_schema,criticScore,0.95,llm_based_matching
7,metacritic,user_score,target_schema,userScore,0.95,llm_based_matching


In [91]:
sales_mapping = matcher.match(sales, df_target)
sales_mapping

[INFO ] PyDI.schemamatching.llm_based - LLM-based schema matching: sales -> target_schema
[INFO ] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO ] PyDI.utils.llm - LLM call: {"timestamp": "2025-12-12T15:32:31.693224Z", "row_index": 0, "attempt": 0, "provider_class": "ChatOpenAI", "model": "gpt-5", "duration_ms": 20840.45910835266, "temperature": 0.0, "max_tokens": null, "usage": {"input_tokens": 4140, "output_tokens": 643, "total_tokens": 4783, "input_token_details": {"audio": 0, "cache_read": 0}, "output_token_details": {"audio": 0, "reasoning": 512}}, "request_messages": [{"type": "system", "content": "You are an expert at aligning table schemas.\n\nGiven Table A and Table B, identify the matching columns between them. For every column in Table A, specify the corresponding column in Table B. If a column in Table A has no match in Table B, map it to null. Represent each mapping as a two-item list like [\"Table A column\", \"Table B column

,source_dataset,source_column,target_dataset,target_column,score,notes
0,sales,Title,target_schema,name,0.95,llm_based_matching
1,sales,Platform,target_schema,platform,0.95,llm_based_matching
2,sales,Year_of_Release,target_schema,releaseYear,0.95,llm_based_matching
3,sales,Genre,target_schema,genres,0.95,llm_based_matching
4,sales,Publisher,target_schema,publisher,0.95,llm_based_matching
5,sales,Global_Sales,target_schema,globalSales,0.95,llm_based_matching
6,sales,Critic_Score,target_schema,criticScore,0.95,llm_based_matching
7,sales,User_Score,target_schema,userScore,0.95,llm_based_matching
8,sales,Developer,target_schema,developer,0.95,llm_based_matching
9,sales,Rating,target_schema,ESRB,0.95,llm_based_matching


## Step 4: Translate and Normalize


In [92]:
# Unify platform names across datasets
platform_groups = {
    "Nintendo Entertainment System": ["NES"],
    "Super Nintendo": ["SNES", "Super Nintendo Entertainment System"],
    "Nintendo 64": ["N64"],
    "GameCube": ["Nintendo GameCube", "GC"],
    "Wii": ["Nintendo Wii"],
    "Game Boy Color": ["GBC"],
    "Game Boy Advance": ["GBA"],
    "Game Boy": ["GB"],
    "DS": ["Nintendo DS"],
    "3DS": ["Nintendo 3DS"],
    "Switch": ["Nintendo Switch"],

    "Playstation": [
        "Playstation (console)", "PlayStation (console)",
        "Playstation 1", "PlayStation 1",
        "PS1", "PSX", "PS"
    ],

    "PS2": ["Playstation 2", "PlayStation 2"],
    "PS3": ["Playstation 3", "PlayStation 3"],
    "PS4": ["Playstation 4", "PlayStation 4"],
    "Playstation Portable": ["PSP"],
    "Playstation Vita": ["PSV", "PS Vita"],
    "Playstation VR": ["PSVR", "PS VR"],

    "Xbox": ["XB", "Xbox (console)"],
    "Xbox One": ["XOne"],
    "Xbox 360": ["X360"],

    "PC": ["Microsoft Windows", "Windows"],
}

platform_map = {}
for canonical, variants in platform_groups.items():
    for alias in variants:
        platform_map[alias.lower()] = canonical

def normalize_platform(series):
    def _norm(x):
        # Leave missing or non-string values as they are
        if not isinstance(x, str):
            return x
        key = x.strip().lower()
        return platform_map.get(key, x.strip())
    
    return series.apply(_norm)

dbpedia["platform"] = normalize_platform(dbpedia["platform"])
metacritic["platform"] = normalize_platform(metacritic["platform"])
sales["Platform"] = normalize_platform(sales["Platform"])

In [93]:
# Create id columns based on index (starting with 1)
sales["id"] = sales.index + 1
metacritic["id"] = metacritic.index + 1
dbpedia["id"] = dbpedia.index + 1

# Make id entries more explicit
dbpedia["id"] = dbpedia["id"].apply(lambda x: f"dbpedia_{x}")
metacritic["id"] = metacritic["id"].apply(lambda x: f"metacritic_{x}")
sales["id"] = sales["id"].apply(lambda x: f"sales_{x}")

In [94]:
# Clean up noisy dbpedia dataset

# Remove suffix (" (video game)") from gameLabel
dbpedia_cleaned = dbpedia.copy()
dbpedia_cleaned["gameLabel"] = dbpedia_cleaned["gameLabel"].str.replace(r" \(video game\)$", "", regex=True)

# drop duplicates that share name, platform, developer and releaseDate
dbpedia_cleaned = dbpedia_cleaned.drop_duplicates(subset=["gameLabel", "platform", "developer", "releaseDate"])

In [95]:
translator = SchemaTranslator()

# Translate + normalize each dataset with its own mapping

spec.set_column("releaseYear", output_type="datetime")
dbpedia_normalized = translator.translate(
    dbpedia_cleaned, dbpedia_mapping,
    normalize=spec, on_failure="keep"
)

# Configure date column to parse special date format (e.g., Oct 26, 2018)
spec.set_column("releaseYear", output_type="datetime", date_format="%b %d, %Y")

metacritic_normalized = translator.translate(
    metacritic, metacritic_mapping,
    normalize=spec, on_failure="keep"
)

# Configure date column to parse year-only values (e.g., "1929", "2010")
spec.set_column("releaseYear", output_type="datetime", date_format="%Y")

sales_normalized = translator.translate(
    sales, sales_mapping,
    normalize=spec, on_failure="keep"
)

[INFO ] root - Translating 6 columns for 'dbpedia'
[WARNING] PyDI.normalization.transform - Column 'publisher' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'criticScore' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'userScore' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'ESRB' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'globalSales' not found in DataFrame
[INFO ] root - Normalization complete: 52920 values transformed, 0 values failed
[INFO ] root - Translating 8 columns for 'metacritic'
[WARNING] PyDI.normalization.transform - Column 'publisher' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'globalSales' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'series' not found in DataFrame
[INFO ] root - Normalization complete: 20492 values transformed, 0 values failed
[INFO ] root - Translating 10 columns for 'sales'
[WARNING] PyDI.normal

In [96]:
# Inspect normalized dbpedia dataset (target columns only)
dbpedia_cols = [c for c in target_columns if c in dbpedia_normalized.columns]
dbpedia_normalized[dbpedia_cols].head(10)

,id,name,releaseYear,developer,platform,series,genres
0,dbpedia_1,San Francisco Rush 2049,2006-02-17,Handheld Games,Game Boy Color,Rush (video game series),Racing video game
1,dbpedia_2,RoboCop (1988 video game),1989-12-12,Ocean Software,Arcade video game,List of RoboCop video games,Beat 'em up
2,dbpedia_3,Air,2016-09-08,Key (company),PlayStation Vita,NaN,Eroge
3,dbpedia_4,Fallout 2,1998-10-29,Black Isle Studios,Mac OS X,Fallout (series),Role-playing video game
4,dbpedia_5,SpongeBob SquarePants: Creature from the Krust...,2006-10-18,Blitz Games,Wii,SpongeBob SquarePants video games,Platform game
5,dbpedia_6,Transformers: Fall of Cybertron,2016-08-08,High Moon Studios,PS4,Transformers,Third-person shooter
6,dbpedia_7,List of Monster Jam video games,2003-12-10,Ubi Soft Barcelona,Xbox One,Monster Jam,Racing game
7,dbpedia_8,Onimusha: Warlords,2002-01-28,Capcom,Xbox One,Onimusha,Hack and slash
8,dbpedia_9,Nicktoons: Battle for Volcano Island,2006-10-24,Halfbrick,PS2,SpongeBob SquarePants video games,Action-adventure game
9,dbpedia_10,Grid Autosport,2019-09-19,Feral Interactive,Xbox 360,Grid (series),Racing video game


In [97]:
metacritic_cols = [c for c in target_columns if c in metacritic_normalized.columns]
metacritic_normalized[metacritic_cols].head(10)

,id,name,releaseYear,developer,platform,criticScore,userScore,ESRB,genres
0,metacritic_1,Red Dead Redemption 2,2018-10-26,Rockstar Games,Xbox One,97.0,8.3,M,"Action Adventure,Open-World"
1,metacritic_2,Grand Theft Auto IV,2008-04-29,Rockstar North,Xbox 360,98.0,8.0,M,"Action Adventure,Modern,Modern,Open-World"
2,metacritic_3,SoulCalibur,1999-09-08,Namco,Dreamcast,98.0,8.4,T,"Action,Fighting,3D"
3,metacritic_4,Tony Hawk's Pro Skater 2,2000-09-20,Neversoft Entertainment,PlayStation,98.0,7.5,T,"Sports,Alternative,Skateboarding"
4,metacritic_5,Super Mario Galaxy,2007-11-12,Nintendo,Wii,97.0,9.1,E,"Action,Platformer,Platformer,3D,3D"
5,metacritic_6,Grand Theft Auto IV,2008-04-29,Rockstar North,PS3,98.0,7.9,M,"Action Adventure,Modern,Modern,Open-World"
6,metacritic_7,Call of Duty 4: Modern Warfare,2007-11-05,Infinity Ward,Xbox 360,94.0,8.5,M,"Action,Shooter,Shooter,First-Person,Modern,Mod..."
7,metacritic_8,The Elder Scrolls IV: Oblivion,2006-03-20,"Bethesda Softworks,Bethesda Game Studios",PC,94.0,8.3,M,"Role-Playing,First-Person,First-Person,Western..."
8,metacritic_9,Super Mario Galaxy 2,2010-05-23,Nintendo EAD Tokyo,Wii,97.0,9.1,E,"Action,Platformer,Platformer,3D,3D"
9,metacritic_10,The Legend of Zelda: Ocarina of Time,1998-11-23,Nintendo,Nintendo 64,99.0,9.0,E,"Action Adventure,Fantasy"


In [98]:
sales_cols = [c for c in target_columns if c in sales_normalized.columns]
sales_normalized[sales_cols].head(10)

,id,name,releaseYear,developer,publisher,platform,criticScore,userScore,ESRB,globalSales,genres
0,sales_1,Wii Sports,2006-01-01,Nintendo,Nintendo,Wii,76.0,8.0,E,82,Sports
1,sales_2,Mario Kart Wii,2008-01-01,Nintendo,Nintendo,Wii,82.0,8.3,E,35,Racing
2,sales_3,Wii Sports Resort,2009-01-01,Nintendo,Nintendo,Wii,80.0,8.0,E,32,Sports
3,sales_4,New Super Mario Bros.,2006-01-01,Nintendo,Nintendo,DS,89.0,8.5,E,29,Platform
4,sales_5,Wii Play,2006-01-01,Nintendo,Nintendo,Wii,58.0,6.6,E,28,Misc
5,sales_6,New Super Mario Bros. Wii,2009-01-01,Nintendo,Nintendo,Wii,87.0,8.4,E,28,Platform
6,sales_7,Mario Kart DS,2005-01-01,Nintendo,Nintendo,DS,91.0,8.6,E,23,Racing
7,sales_8,Wii Fit,2007-01-01,Nintendo,Nintendo,Wii,80.0,7.7,E,22,Sports
8,sales_9,Kinect Adventures!,2010-01-01,Good Science Studio,Microsoft Game Studios,Xbox 360,61.0,6.3,E,21,Misc
9,sales_10,Wii Fit Plus,2009-01-01,Nintendo,Nintendo,Wii,80.0,7.4,E,21,Sports


In [99]:
# convert genre strings to lists
normalized_sets = [dbpedia_normalized, metacritic_normalized, sales_normalized]
for s in normalized_sets:
    s["genres"] = s["genres"].apply(lambda x: [g.strip() for g in x.split(",")] if isinstance(x, str) else x)


In [100]:
# Only keep target columns
dbpedia = dbpedia_normalized[dbpedia_cols].copy()
metacritic = metacritic_normalized[metacritic_cols].copy()
sales = sales_normalized[sales_cols].copy()

## Part 2: Data Profiling

In [101]:
# Display basic information
datasets = [dbpedia, metacritic, sales]
names = ["DBpedia", "Metacritic", "Sales"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Total records across all datasets: 82,701


In [102]:
from PyDI.utils import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

dbpedia:
  Rows: 54,329
  Columns: 7
  Total nulls: 29,378
  Null percentage: 7.7%
  Null counts per column:
    releaseYear: 1,409 (2.6%)
    developer: 1,406 (2.6%)
    platform: 460 (0.8%)
    series: 25,801 (47.5%)
    genres: 302 (0.6%)

metacritic:
  Rows: 20,494
  Columns: 9
  Total nulls: 3,720
  Null percentage: 2.0%
  Null counts per column:
    releaseYear: 2 (0.0%)
    developer: 19 (0.1%)
    criticScore: 10 (0.0%)
    userScore: 1,413 (6.9%)
    ESRB: 2,276 (11.1%)

sales:
  Rows: 7,878
  Columns: 11
  Total nulls: 1
  Null percentage: 0.0%
  Null counts per column:
    publisher: 1 (0.0%)



### Attribute Coverage Analysis

In [103]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

[INFO ] PyDI.fusion.analysis - Analyzed 12 attributes across 3 datasets


📊 Attribute coverage across datasets:


,attribute,dbpedia_count,dbpedia_pct,dbpedia_coverage,dbpedia_samples,metacritic_count,metacritic_pct,metacritic_coverage,metacritic_samples,sales_count,sales_pct,sales_coverage,sales_samples,avg_coverage,max_coverage,datasets_with_attribute
0,ESRB,0/0,0%,0.000000,N/A,18218/20494,88.9%,0.888943,"['M', 'M', 'T']",7878/7878,100.0%,1.000000,"['E', 'E', 'E']",0.629648,1.000000,2
1,criticScore,0/0,0%,0.000000,N/A,20484/20494,100.0%,0.999512,"[97.0, 98.0, 98.0]",7878/7878,100.0%,1.000000,"[76.0, 82.0, 80.0]",0.666504,1.000000,2
2,developer,52923/54329,97.4%,0.974121,"['Handheld Games', 'Ocean Software', 'Key (com...",20475/20494,99.9%,0.999073,"['Rockstar Games', 'Rockstar North', 'Namco']",7878/7878,100.0%,1.000000,"['Nintendo', 'Nintendo', 'Nintendo']",0.991065,1.000000,3
3,genres,54027/54329,99.4%,0.994441,"[['Racing video game'], [""Beat 'em up""], ['Ero...",20494/20494,100.0%,1.000000,"[['Action Adventure', 'Open-World'], ['Action ...",7878/7878,100.0%,1.000000,"[['Sports'], ['Racing'], ['Sports']]",0.998147,1.000000,3
4,globalSales,0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,7878/7878,100.0%,1.000000,"[82, 35, 32]",0.333333,1.000000,1
5,id,54329/54329,100.0%,1.000000,"['dbpedia_1', 'dbpedia_2', 'dbpedia_3']",20494/20494,100.0%,1.000000,"['metacritic_1', 'metacritic_2', 'metacritic_3']",7878/7878,100.0%,1.000000,"['sales_1', 'sales_2', 'sales_3']",1.000000,1.000000,3
6,name,54329/54329,100.0%,1.000000,"['San Francisco Rush 2049', 'RoboCop (1988 vid...",20494/20494,100.0%,1.000000,"['Red Dead Redemption 2', 'Grand Theft Auto IV...",7878/7878,100.0%,1.000000,"['Wii Sports', 'Mario Kart Wii', 'Wii Sports R...",1.000000,1.000000,3
7,platform,53869/54329,99.2%,0.991533,"['Game Boy Color', 'Arcade video game', 'PlayS...",20494/20494,100.0%,1.000000,"['Xbox One', 'Xbox 360', 'Dreamcast']",7878/7878,100.0%,1.000000,"['Wii', 'Wii', 'Wii']",0.997178,1.000000,3
8,publisher,0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,7877/7878,100.0%,0.999873,"['Nintendo', 'Nintendo', 'Nintendo']",0.333291,0.999873,1
9,releaseYear,52920/54329,97.4%,0.974065,"[Timestamp('2006-02-17 00:00:00'), Timestamp('...",20492/20494,100.0%,0.999902,"[Timestamp('2018-10-26 00:00:00'), Timestamp('...",7878/7878,100.0%,1.000000,"[Timestamp('2006-01-01 00:00:00'), Timestamp('...",0.991323,1.000000,3



🔗 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['ESRB', 'criticScore', 'developer', 'genres', 'id', 'name', 'platform', 'releaseYear', 'userScore']


## Part 3: Entity Matching

### Step 1: Blocking

In [104]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [105]:
# Import blocking methods
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker
import re

def get_longest_token(name):
    tokens = re.split(r"[^A-Za-z0-9_']+", str(name))    
    tokens = [t for t in tokens if t]
    return max(tokens, key=len) if tokens else ''

dbpedia['name_longest_token'] = dbpedia['name'].apply(get_longest_token)
metacritic['name_longest_token'] = metacritic['name'].apply(get_longest_token)
sales['name_longest_token'] = sales['name'].apply(get_longest_token)

standard_blocker_m2d = StandardBlocker(
    metacritic, dbpedia,
    on=['name_longest_token', 'platform'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 13482 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 19214 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 6063 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv


### Step 2: Evaluate Blocking Against Ground Truth

In [106]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "metacritic_2_dbpedia_test.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 8 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 26 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 43 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 63 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 90 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 125 true matches
[INFO ] root - Processed 70 batches, 69971 pairs, 185 true matches
[INFO ] root -   Pair Completeness: 0.989
[INFO ] root -   Pair Quality:      0.003
[INFO ] root -   Reduction Ratio:   0.999937
[INFO ] root -   True Matches Found: 185/187
[INFO ] root -   Batches Processed:  70
[INFO ] root - Blocking evaluation complete!


{'pair_completeness': 0.9893048128342246,
 'pair_quality': 0.0026439524946049076,
 'reduction_ratio': 0.9999371566052063,
 'total_candidates': 69971,
 'total_possible_pairs': 1113418526,
 'true_positives_found': 185,
 'total_true_pairs': 187,
 'batches_processed': 70,
 'evaluation_timestamp': '2025-12-12T16:32:58.686740',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/blocking_detailed_results.csv']}

In [107]:
# def get_first_token(name):
#     return re.split(r"[^A-Za-z0-9_']+", str(name))[0]
# metacritic['name_first_token'] = metacritic['name'].apply(get_first_token)
# sales['name_first_token'] = sales['name'].apply(get_first_token)

standard_blocker_m2s = StandardBlocker(
    metacritic, sales,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "metacritic_2_sales_test.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2s,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 5497 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 2231 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 2100 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 2 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 5 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 8 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 10 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 12 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 16 true matches
[INFO ] root - Proc

{'pair_completeness': 0.9555555555555556,
 'pair_quality': 0.0007636073045846035,
 'reduction_ratio': 0.9989536501225023,
 'total_candidates': 168935,
 'total_possible_pairs': 161451732,
 'true_positives_found': 129,
 'total_true_pairs': 135,
 'batches_processed': 169,
 'evaluation_timestamp': '2025-12-12T16:33:10.672728',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/blocking_detailed_results.csv']}

Now let's evaluate which blocking method we want to use for each dataset combination:

### Step 3: Entity Matching with Comparators

In [108]:
from PyDI.entitymatching import StringComparator, DateComparator

# Create comparators for different attributes
comparators_m2d = [
    # Name similarity - most important for games
    StringComparator(
        column='name',
        similarity_function='jaccard',  # Good for game names
        preprocess=str.lower  # Case normalization
    ),
    
    # Platform similarity - supporting evidence
    StringComparator(
        column='developer',
        similarity_function='jaccard',
        preprocess=str.lower
    ),

    # Date proximity - games from same year likely same game
    DateComparator(
        column='releaseYear',
    )
]

comparators_m2s = [
    StringComparator(
        column='name',
        similarity_function='jaccard',
        preprocess=str.lower
    ),
        # Platform similarity - supporting evidence
    StringComparator(
        column='platform',
        similarity_function='jaccard',
    ),
    DateComparator(
        column='releaseYear',
        max_days_difference=360  # Allow almost 1 year difference
    )
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [109]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_m2d = matcher.match(
    df_left=metacritic,
    df_right=dbpedia, 
    candidates=standard_blocker_m2d, # pass the blocker, which will internally generate candidate pairs using batching
    comparators=comparators_m2d,
    weights=[0.6, 0.3, 0.1], # name, developer, releaseYear
    threshold=0.9, # set a similarity threshold for a match
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 20494 x 54329 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 20494 x 54329 elements after 0:00:0.127; 69971 blocked pairs (reduction ratio: 0.9999371566052063)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:29.495; found 4691 correspondences.


In [110]:
correspondences_m2s = matcher.match(
    df_left=metacritic,
    df_right=sales, 
    candidates=standard_blocker_m2s,
    comparators=comparators_m2s,
    weights=[0.6, 0.3, 0.1], # name, platform, releaseYear
    threshold=0.8,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 20494 x 7878 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 20494 x 7878 elements after 0:00:0.132; 168935 blocked pairs (reduction ratio: 0.9989536501225023)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:67.223; found 6546 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [111]:
gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "metacritic_2_dbpedia_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  180
[INFO ] root -   True Negatives:  391
[INFO ] root -   False Positives: 1
[INFO ] root -   False Negatives: 7
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.986
[INFO ] root -   Precision: 0.994
[INFO ] root -   Recall:    0.963
[INFO ] root -   F1-Score:  0.978


{'precision': 0.994475138121547,
 'recall': 0.9625668449197861,
 'f1': 0.9782608695652174,
 'accuracy': 0.9861830742659758,
 'true_positives': 180,
 'false_positives': 1,
 'false_negatives': 7,
 'true_negatives': 391,
 'threshold_used': 0.0,
 'total_correspondences': 4691,
 'filtered_correspondences': 4691,
 'evaluation_timestamp': '2025-12-12T16:34:57.230776',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/games/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/games/debug_results_entity_matching/matching_detailed_results.csv']}

In [112]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

Analyzing cluster size distribution in our entity matching results...


[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 3199 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	2295	|	71.74%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	554	|	17.32%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	202	|	6.31%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	93	|	2.91%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	35	|	1.09%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	11	|	0.34%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	6	|	0.19%
[INFO ] PyDI.entitymatching.evaluation - 		9	|	1	|	0.03%
[INFO ] PyDI.entitymatching.evaluation - 		10	|	1	|	0.03%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	1	|	0.03%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/output/games/cluster_analysis/cluster_size_distribution.csv


In [113]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2d,
    out_path=cluster_details_path
)

[INFO ] root - Cluster details written to /Users/luca/PycharmProjects/PyDI/usecases/output/games/cluster_analysis/detailed_cluster_info.json
[INFO ] root - Exported 3199 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [114]:
from PyDI.entitymatching import MaximumBipartiteMatching
     
clusterer = MaximumBipartiteMatching()
correspondences_m2d = clusterer.cluster(correspondences_m2d)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

[INFO ] root - Filtered correspondences: 4691 -> 4691 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 4691 -> 3199 
[INFO ] root - MaximumBipartiteMatching: 4691 -> 3199 correspondences
[INFO ] root - MaximumBipartiteMatching: 7890 -> 6398 entities
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 3199 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	3199	|	100.00%
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  160
[INFO ] root -   True Negatives:  392
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 27
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.953
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.856
[INFO ] root -   F1-Score:  0.922


In [115]:
from PyDI.entitymatching import  MaximumBipartiteMatching

gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "metacritic_2_sales_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2s,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2s,
)

clusterer = MaximumBipartiteMatching()
correspondences_m2s = clusterer.cluster(correspondences_m2s)


cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2s,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2s,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  122
[INFO ] root -   True Negatives:  422
[INFO ] root -   False Positives: 25
[INFO ] root -   False Negatives: 13
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.935
[INFO ] root -   Precision: 0.830
[INFO ] root -   Recall:    0.904
[INFO ] root -   F1-Score:  0.865
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 6305 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	6197	|	98.29%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	27	|	0.43%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	74	|	1.17%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	2	|	0.03%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	5	|	0.08%
[INFO ] root - Filtered correspondences: 6546 -> 6546 (threshold=0.0)
[INFO ] root - Maximum bip

## Part 4: Data Fusion

In [116]:
# We are only interested in year of release --> change dates to YYYY-01-01
dbpedia["releaseYear"] = pd.to_datetime({
    "year": dbpedia["releaseYear"].dt.year,
    "month": 1,
    "day": 1
})

metacritic["releaseYear"] = pd.to_datetime({
    "year": metacritic["releaseYear"].dt.year,
    "month": 1,
    "day": 1
})

sales["releaseYear"] = pd.to_datetime({
    "year": sales["releaseYear"].dt.year,
    "month": 1,
    "day": 1
})

In [117]:
metacritic["metacritic_id"] = metacritic["id"]

# Assign trust scores to datasets
metacritic.attrs["trust_score"] = 3
sales.attrs["trust_score"] = 2
dbpedia.attrs["trust_score"] = 1

all_correspondences = pd.concat([correspondences_m2d, correspondences_m2s], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 9,590


## Step 1: Define Fusion Strategy 

In [118]:
from PyDI.fusion import DataFusionStrategy, prefer_higher_trust, voting, average, union

strategy = DataFusionStrategy('game_fusion_strategy')
['ESRB', 'criticScore', 'developer', 'id', 'name', 'platform', 'releaseYear', 'userScore']
strategy.add_attribute_fuser('name', voting)
strategy.add_attribute_fuser('platform', voting)
strategy.add_attribute_fuser('developer', voting)
strategy.add_attribute_fuser('releaseYear', voting, trust_key="trust_score")
strategy.add_attribute_fuser('ESRB', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('criticScore', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('userScore', average)
strategy.add_attribute_fuser('genres', union)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'name' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'platform' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'developer' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'releaseYear' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'ESRB' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'criticScore' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'userScore' using rule 'average'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'genres' using rule 'union'


## Step 2: Run Fusion

In [119]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[metacritic, dbpedia, sales],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/games/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'game_fusion_strategy'


[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 17600 of 17600 unique IDs
[INFO ] PyDI.fusion.engine - Created 73111 record groups from 9590 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 73111 clusters:
[INFO ] PyDI.fusion.engine - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.fusion.engine - 	──────────────────────────────────────────────────
[INFO ] PyDI.fusion.engine - 		2	|	6430	|	8.79%
[INFO ] PyDI.fusion.engine - 		3	|	1580	|	2.16%
[INFO ] PyDI.fusion.engine - Attribute Consistencies:
[INFO ] PyDI.fusion.engine -     ESRB: 1.00
[INFO ] PyDI.fusion.engine -     _id: 0.00
[INFO ] PyDI.fusion.engine -     criticScore: 0.99
[INFO ] PyDI.fusion.engine -     developer: 0.92
[INFO ] PyDI.fusion.engine -     genres: 0.00
[INFO ] PyDI.fusion.engine -     globalSales: 1.00
[INFO ] PyDI.fusion.engine -     id: 0.00
[INFO ] PyDI.fusion.engine -     metacritic_id: 1.00
[INFO ] PyDI.

Fused rows: 8,010


,_id,_fusion_sources,_fusion_source_datasets,publisher,ESRB,releaseYear,name,developer,userScore,globalSales,name_longest_token,platform,id,genres,metacritic_id,criticScore,_fusion_confidence,_fusion_metadata,series
0,sales_1373,"[sales_1373, metacritic_4251]","[sales, metacritic]",Electronic Arts,E,2004-01-01,FIFA Soccer 2005,EA Canada,8.55,0.0,Soccer,Xbox,sales_1373,"[Sim, Soccer, Sports, Traditional]",metacritic_4251,81.0,0.831871,"{'publisher_rule': 'first_non_null', 'publishe...",NaN
1,sales_924,"[sales_924, metacritic_15398]","[sales, metacritic]",Sony Computer Entertainment,T,2008-01-01,SingStar Abba,SCEE London Studio,7.60,1.0,SingStar,PS2,sales_924,"[Misc, Miscellaneous, Music, Rhythm]",metacritic_15398,64.0,0.791667,"{'publisher_rule': 'first_non_null', 'publishe...",NaN
2,sales_3709,"[sales_3709, metacritic_18544]","[sales, metacritic]",Midas Interactive Entertainment,E,2000-01-01,Real Pool,Takara,NaN,0.0,Real,PS2,sales_3709,"[Billiards, Miscellaneous, Parlor, Sports]",metacritic_18544,54.0,0.750000,"{'publisher_rule': 'first_non_null', 'publishe...",NaN
3,metacritic_3453,"[metacritic_3453, sales_6944]","[metacritic, sales]",Atari,E,2002-01-01,V-Rally 3,Velez & Dubail,7.90,0.0,Rally,Game Boy Advance,metacritic_3453,"[Driving, Racing, Rally / Offroad]",metacritic_3453,82.0,0.833333,"{'publisher_rule': 'first_non_null', 'publishe...",NaN
4,metacritic_11015,"[metacritic_11015, dbpedia_57116]","[metacritic, dbpedia]",NaN,E,2002-01-01,Aero the Acro-bat,Atomic Planet Entertainment,7.50,NaN,Aero,Game Boy Advance,metacritic_11015,"[2D, Action, Platform game, Platformer]",metacritic_11015,71.0,0.772727,"{'ESRB_rule': 'prefer_higher_trust', 'ESRB_sou...",None


## Step 3: Evaluate Data Fusion

In [120]:
from PyDI.fusion import tokenized_match, year_only_match, boolean_match, numeric_tolerance_match, exact_match
strategy.add_evaluation_function("name", exact_match)
strategy.add_evaluation_function("platform", exact_match)
strategy.add_evaluation_function("developer", exact_match)
strategy.add_evaluation_function("releaseYear", year_only_match)
strategy.add_evaluation_function("ESRB", exact_match)
strategy.add_evaluation_function("criticScore", numeric_tolerance_match, tolerance=2)
strategy.add_evaluation_function("userScore", numeric_tolerance_match, tolerance=0.2)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'platform'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'developer'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'releaseYear'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'ESRB'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'criticScore' with params {'tolerance': 2}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'userScore' with params {'tolerance': 0.2}


In [121]:
from PyDI.io import load_xml
from PyDI.fusion import DataFusionEvaluator

# Finally, evaluate against test set
fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set.xml', name='fusion_test_set', nested_handling='aggregate')
fusion_test_set['releaseYear'] = pd.to_datetime(fusion_test_set['releaseYear'],errors='coerce')

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the test set
print("Evaluating fusion results against test set...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='metacritic_id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Test Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/games/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation


Evaluating fusion results against test set...


[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.832 overall accuracy (99/119)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 20 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	userScore                        |      10 |     50.00%%
[INFO ] PyDI.fusion.evaluation - 	name                             |       3 |     15.00%%
[INFO ] PyDI.fusion.evaluation - 	developer                        |       3 |     15.00%%
[INFO ] PyDI.fusion.evaluation - 	publisher                        |       2 |     10.00%%
[INFO ] PyDI.fusion.evaluation - 	ESRB                             |       2 |     10.00%%



Fusion Test Results:
  overall_accuracy: 0.832
  macro_accuracy: 0.827
  num_evaluated_records: 15
  num_evaluated_attributes: 8
  total_evaluations: 119
  total_correct: 99
  publisher_accuracy: 0.867
  publisher_count: 15
  ESRB_accuracy: 0.867
  ESRB_count: 15
  releaseYear_accuracy: 1.000
  releaseYear_count: 15
  name_accuracy: 0.800
  name_count: 15
  developer_accuracy: 0.800
  developer_count: 15
  userScore_accuracy: 0.286
  userScore_count: 14
  platform_accuracy: 1.000
  platform_count: 15
  criticScore_accuracy: 1.000
  criticScore_count: 15

Overall Accuracy: 83.2%
